In [68]:
import torch
import matplotlib.pyplot as plt
%matplotlib


Using matplotlib backend: TkAgg


In [69]:
pos_x = (0.6, 0.6)
pos_y = (0., 0.)
pos_z = (0.0, 0.4)
roll = (-torch.pi/6, -torch.pi/6)
# pitch = (-torch.pi/6, -torch.pi/6)
pitch = (0, 0)
yaw = (0, 0)

In [70]:
num_envs = 10

In [71]:
env_ids = torch.arange(num_envs)

In [72]:
pose_command_o = torch.zeros(num_envs, 7)
pose_command_o[:, 3] = 1.0
pose_command_w = torch.zeros_like(pose_command_o)

In [73]:
@torch.jit.script
def quat_from_euler_xyz(roll: torch.Tensor, pitch: torch.Tensor, yaw: torch.Tensor) -> torch.Tensor:
    """Convert rotations given as Euler angles in radians to Quaternions.

    Note:
        The euler angles are assumed in XYZ convention.

    Args:
        roll: Rotation around x-axis (in radians). Shape is (N,).
        pitch: Rotation around y-axis (in radians). Shape is (N,).
        yaw: Rotation around z-axis (in radians). Shape is (N,).

    Returns:
        The quaternion in (w, x, y, z). Shape is (N, 4).
    """
    cy = torch.cos(yaw * 0.5)
    sy = torch.sin(yaw * 0.5)
    cr = torch.cos(roll * 0.5)
    sr = torch.sin(roll * 0.5)
    cp = torch.cos(pitch * 0.5)
    sp = torch.sin(pitch * 0.5)
    # compute quaternion
    qw = cy * cr * cp + sy * sr * sp
    qx = cy * sr * cp - sy * cr * sp
    qy = cy * cr * sp + sy * sr * cp
    qz = sy * cr * cp - cy * sr * sp

    return torch.stack([qw, qx, qy, qz], dim=-1)

In [74]:
r = torch.empty(len(env_ids))
pose_command_o[env_ids, 0] = r.uniform_(*pos_x)
pose_command_o[env_ids, 1] = r.uniform_(*pos_y)
pose_command_o[env_ids, 2] = r.uniform_(*pos_z)
# -- orientation
euler_angles = torch.zeros_like(pose_command_o[env_ids, :3])
euler_angles[:, 0].uniform_(*roll)
euler_angles[:, 1].uniform_(*pitch)
euler_angles[:, 2].uniform_(*yaw)
pose_command_o[env_ids, 3:] = quat_from_euler_xyz(
    euler_angles[:, 0], euler_angles[:, 1], euler_angles[:, 2]
)

In [75]:
pose_command_o

tensor([[ 0.6000,  0.0000,  0.2085,  0.9659, -0.2588,  0.0000,  0.0000],
        [ 0.6000,  0.0000,  0.1810,  0.9659, -0.2588,  0.0000,  0.0000],
        [ 0.6000,  0.0000,  0.2762,  0.9659, -0.2588,  0.0000,  0.0000],
        [ 0.6000,  0.0000,  0.1676,  0.9659, -0.2588,  0.0000,  0.0000],
        [ 0.6000,  0.0000,  0.1539,  0.9659, -0.2588,  0.0000,  0.0000],
        [ 0.6000,  0.0000,  0.2866,  0.9659, -0.2588,  0.0000,  0.0000],
        [ 0.6000,  0.0000,  0.1278,  0.9659, -0.2588,  0.0000,  0.0000],
        [ 0.6000,  0.0000,  0.1267,  0.9659, -0.2588,  0.0000,  0.0000],
        [ 0.6000,  0.0000,  0.1420,  0.9659, -0.2588,  0.0000,  0.0000],
        [ 0.6000,  0.0000,  0.0115,  0.9659, -0.2588,  0.0000,  0.0000]])

In [76]:
w = 0.65
l = 0.65

In [77]:
corners = torch.tensor([
    [w / 2, l / 2, 0],
    [-w / 2, l / 2, 0],
    [-w / 2, -l / 2, 0],
    [w / 2, -l / 2, 0]
], dtype=torch.float32)

In [78]:
def euler_to_rotation_matrix(roll, pitch, yaw):
    # Create individual rotation matrices
    R_x = torch.tensor([
        [1, 0, 0],
        [0, torch.cos(roll), -torch.sin(roll)],
        [0, torch.sin(roll), torch.cos(roll)]
    ], dtype=torch.float32)
    
    R_y = torch.tensor([
        [torch.cos(pitch), 0, torch.sin(pitch)],
        [0, 1, 0],
        [-torch.sin(pitch), 0, torch.cos(pitch)]
    ], dtype=torch.float32)
    
    R_z = torch.tensor([
        [torch.cos(yaw), -torch.sin(yaw), 0],
        [torch.sin(yaw), torch.cos(yaw), 0],
        [0, 0, 1]
    ], dtype=torch.float32)
    
    # Combine the rotation matrices
    R = torch.mm(R_x, torch.mm(R_y, R_z))
    return R


In [79]:
ground_contact = torch.zeros(num_envs, dtype=torch.bool)

In [80]:
fig = plt.figure(figsize=(10, 10))
ax = plt.axes(projection='3d')
ax.set_zlim(0, 0.6)
ax.set_xlabel('x')
ax.set_ylabel('y')

# for i in range(1):
for i in range(num_envs):
    # Extract position and Euler angles for the current environment
    pos = pose_command_o[i, :3]
    roll, pitch, yaw = euler_angles[i, 0], euler_angles[i, 1], euler_angles[i, 2]

    # Get the rotation matrix from Euler angles
    r = euler_to_rotation_matrix(roll, pitch, yaw)
    # r = euler_to_rotation_matrix(pitch, roll, yaw)

    normal_cornes = corners + pos
    # Transform corners
    transformed_corners = torch.mm(corners, r.T) + pos

    # ax.scatter(pos[..., 0], pos[..., 1], pos[..., 2], color='black', alpha=1)
    # ax.scatter(normal_cornes[..., 0], normal_cornes[...,I 1], normal_cornes[..., 2], color='blue', alpha=1)
    ax.scatter(transformed_corners[..., 0], transformed_corners[..., 1], transformed_corners[..., 2], color='red', alpha=1)

    if (transformed_corners[:, 2] < 0).any():
        ground_contact[i] = True

plt.show()


In [81]:
ground_contact

tensor([False, False, False, False,  True, False,  True,  True,  True,  True])